In [ ]:
!pip install pyTelegramBotAPI --quiet
!pip install python-dotenv --quiet
import telebot
import requests
import logging
import time
import os
from threading import Thread
from dotenv import load_dotenv
from telebot import types

# Загрузка конфигурации
load_dotenv()

class Config:
    YC_API_KEY = os.getenv('YC_API_KEY', )
    YC_FOLDER_ID = os.getenv('YC_FOLDER_ID', "")
    TG_TOKEN = os.getenv('TG_TOKEN', "")
    BOT_VERSION = "2.3"
    PRODUCT_TEMPLATES = {
        "одежда": "Напиши продающее описание для одежды. Характеристики: {features}",
        "электроника": "Создай описание для электронного устройства. Параметры: {features}",
        "книги": "Составь аннотацию для книги. Детали: {features}",
        "другое": "Напиши продающий текст для товара. Особенности: {features}"
    }
    @staticmethod
    def model_uri():
        return f"gpt://{Config.YC_FOLDER_ID}/yandexgpt-lite"

# Инициализация бота
bot = telebot.TeleBot(
    Config.TG_TOKEN,
    threaded=False,
    parse_mode='HTML',
    skip_pending=True
)

# Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('product_bot.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

def send_long_message(chat_id, text, **kwargs):
    """Безопасно отправить длинное сообщение в Telegram"""
    for i in range(0, len(text), 4000):
        bot.send_message(chat_id, text[i:i+4000], **kwargs)

class ProductDescriptionGenerator:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            "Authorization": f"Api-Key {Config.YC_API_KEY}",
            "x-folder-id": Config.YC_FOLDER_ID,
            "Content-Type": "application/json"
        })

    def generate_description(self, product_type: str, features: str) -> str:
        try:
            template = Config.PRODUCT_TEMPLATES.get(product_type.lower(), Config.PRODUCT_TEMPLATES["другое"])
            prompt = template.format(features=features)
            data = {
                "modelUri": Config.model_uri(),
                "completionOptions": {"temperature": 0.7, "maxTokens": 1500},
                "messages": [{"role": "user", "text": prompt}]
            }
            response = self.session.post(
                "https://llm.api.cloud.yandex.net/foundationModels/v1/completion",
                json=data,
                timeout=20
            )
            response.raise_for_status()
            result = response.json()
            alt = result.get("result", {}).get("alternatives")
            if not alt or "message" not in alt[0] or "text" not in alt[0]["message"]:
                raise ValueError("Некорректный ответ YandexGPT: " + str(result))
            return alt[0]["message"]["text"]
        except Exception as e:
            logger.error(f"Description generation failed: {str(e)}")
            return None

    def notify_admin(self, message: str):
        if Config.ADMIN_ID:
            try:
                bot.send_message(Config.ADMIN_ID, f"🔔 {message}")
            except Exception as e:
                logger.error(f"Admin notification failed: {str(e)}")

generator = ProductDescriptionGenerator()

def create_main_keyboard():
    keyboard = types.ReplyKeyboardMarkup(resize_keyboard=True, row_width=2)
    buttons = [
        types.KeyboardButton("👕 Одежда"),
        types.KeyboardButton("📱 Электроника"),
        types.KeyboardButton("📚 Книги"),
        types.KeyboardButton("ℹ️ Помощь")
    ]
    keyboard.add(*buttons)
    return keyboard

def create_cancel_keyboard():
    keyboard = types.ReplyKeyboardMarkup(resize_keyboard=True)
    keyboard.add(types.KeyboardButton("❌ Отмена"))
    return keyboard

user_sessions = {}

@bot.message_handler(commands=['start', 'help'])
def send_welcome(message):
    welcome_text = f"""
    🛍️ <b>Генератор описаний товаров v{Config.BOT_VERSION}</b> 🛍️

    Я помогу создать продающие описания для:
    • Одежды и аксессуаров
    • Электроники и техники
    • Книг и печатной продукции
    • Любых других товаров

    <b>Как работать:</b>
    1. Выберите тип товара
    2. Укажите характеристики
    3. Получите готовое описание!
    """
    bot.send_message(
        message.chat.id,
        welcome_text,
        reply_markup=create_main_keyboard()
    )

@bot.message_handler(func=lambda m: m.text in ["👕 Одежда", "📱 Электроника", "📚 Книги"])
def handle_product_type(message):
    try:
        product_type = message.text.split()[1].lower()
        user_sessions[message.chat.id] = {"type": product_type}
        example = {
            "одежда": "Например: 'Женское платье, хлопок 95%, эластан 5%, размеры S-XXL, 5 цветов'",
            "электроника": "Например: 'Смартфон, экран 6.5\", 128 ГБ памяти, процессор Snapdragon 888'",
            "книги": "Например: 'Роман, 320 страниц, твердый переплет, жанр - фэнтези'"
        }.get(product_type, "Укажите основные характеристики товара")
        bot.send_message(
            message.chat.id,
            f"📝 <b>Введите характеристики товара ({product_type}):</b>\n\n{example}",
            reply_markup=create_cancel_keyboard()
        )
    except Exception as e:
        logger.error(f"Product type error: {str(e)}")
        bot.reply_to(message, "⚠️ Произошла ошибка. Попробуйте еще раз.")

@bot.message_handler(func=lambda m: m.chat.id in user_sessions and m.text != "❌ Отмена")
def handle_product_features(message):
    try:
        chat_id = message.chat.id
        product_type = user_sessions[chat_id]["type"]
        features = message.text
        bot.send_chat_action(chat_id, 'typing')
        description = generator.generate_description(product_type, features)
        if description:
            result = f"🛒 <b>Описание товара ({product_type}):</b>\n\n{description}"
            send_long_message(chat_id, result, reply_markup=create_main_keyboard())
        else:
            bot.send_message(
                chat_id,
                "⚠️ Не удалось сгенерировать описание. Попробуйте другие характеристики.",
                reply_markup=create_main_keyboard()
            )
        del user_sessions[chat_id]
    except Exception as e:
        logger.error(f"Description generation error: {str(e)}")
        generator.notify_admin(f"Generation error: {str(e)}")
        bot.reply_to(message, "⚠️ Произошла ошибка генерации. Попробуйте еще раз.")

@bot.message_handler(func=lambda m: m.text == "❌ Отмена")
def cancel_action(message):
    if message.chat.id in user_sessions:
        del user_sessions[message.chat.id]
    bot.send_message(
        message.chat.id,
        "Действие отменено. Что будем делать дальше?",
        reply_markup=create_main_keyboard()
    )

def run_bot():
    logger.info(f"Starting Product Description Bot v{Config.BOT_VERSION}")
    while True:
        try:
            bot.infinity_polling()
        except Exception as e:
            logger.error(f"Bot crashed: {str(e)}")
            generator.notify_admin(f"Bot crashed: {str(e)}")
            time.sleep(10)

if __name__ == "__main__":
    logger.info("Testing services connection...")
    test_desc = generator.generate_description("электроника", "Смартфон, экран 6.5 дюймов, 128 ГБ памяти")
    logger.info(f"Description test: {'OK' if test_desc else 'FAILED'}")
    if test_desc:
        logger.info("Service is working!")
        generator.notify_admin(f"Product Bot v{Config.BOT_VERSION} started successfully")
        Thread(target=run_bot, daemon=True).start()
        while True:
            time.sleep(1)
    else:
        logger.error("Service check failed!")
        generator.notify_admin("❌ Service check failed on startup!")

KeyboardInterrupt: 